# Volume 1. Projection Parameter Lab

Question: which projection parameters make a sparse winner set settle quickly, and which settings leave more churn?

This is a small teaching sweep, not a benchmark. Every row is an independent brain with one stimulus, one area, and one traced projection.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from neural_assemblies.assembly_calculus.tracing import ProjectionSweepConfig, project_trace, projection_sweep
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import plot_parameter_heatmap, plot_trace_metrics, plot_winner_turnover


Sweep a small grid of `beta` and projection rounds. `final_overlap_prev` close to 1 means the last two observed assemblies were nearly the same.


In [ ]:
N = 3_000
K = 50
BETAS = [0.03, 0.08, 0.13]
ROUNDS = [3, 5, 8]

configs = [
    ProjectionSweepConfig(n=N, k=K, beta=beta, rounds=rounds, seed=11)
    for beta in BETAS
    for rounds in ROUNDS
]
results = pd.DataFrame(projection_sweep(configs))
results


In [ ]:
pivot = results.pivot(index="beta", columns="rounds", values="final_overlap_prev")
ax, values = plot_parameter_heatmap(
    pivot.values,
    x_labels=pivot.columns,
    y_labels=pivot.index,
    title="Final consecutive overlap",
    cbar_label="overlap",
)
ax.set_xlabel("projection rounds")
ax.set_ylabel("beta")
plt.show()
plt.close(ax.figure)


A single grid hides the process. Compare a short, low-plasticity trace with a longer, stronger one.


In [ ]:
def traced_projection(beta, rounds):
    brain = Brain(p=0.05, save_winners=True, seed=11, engine="numpy_sparse")
    brain.add_stimulus("cue", K)
    brain.add_area("A", N, K, beta)
    return project_trace(brain, "cue", "A", rounds=rounds)

low_trace = traced_projection(beta=0.03, rounds=3)
high_trace = traced_projection(beta=0.13, rounds=8)

pd.DataFrame([low_trace.summary(), high_trace.summary()])


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
plot_trace_metrics(low_trace, axes=axes[0], title="Low beta, short run", color="#e15554")
plot_trace_metrics(high_trace, axes=axes[1], title="Higher beta, longer run", color="#4d9de0")
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_winner_turnover(low_trace, ax=axes[0], title="Low beta turnover")
plot_winner_turnover(high_trace, ax=axes[1], title="Higher beta turnover")
plt.show()
plt.close(fig)


The intended habit: do not call a projection stable because a final dot plot looks tidy. Check consecutive overlap, winner turnover, and the parameter regime that produced the trace.
